# Session 5 Practice: MCP Server and Client

Work through these activities after the [Session 5 lesson](<../../Session 5 Introduction to MCP.ipynb>). Use the same Python environment. The first activities run locally without keys; live weather and the chatbot are optional.

Run this practice notebook from `practice/session5/`. Keep the HTTP server running in Terminal 1 and run the chatbot in Terminal 2.

## 1. Add a tool

Run the starter server below. Add a `celsius_to_fahrenheit(celsius: float)` tool using `$F = C \times 9/5 + 32$`. Include a docstring, discover its generated schema, and call it through the client with `0` and `100` degrees Celsius. Expect `32` and `212` degrees Fahrenheit.


In [ ]:
from fastmcp import FastMCP, Client

practice_server = FastMCP(name="Practice Calculator")

@practice_server.tool()
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

# Add your temperature conversion tool here before creating the client.

async with Client(practice_server) as client:
    tools = await client.list_tools()  # Inspect registered tool names and schemas.
    for tool in tools:
        print(tool)
    result = await client.call_tool("add", {"a": 3, "b": 7})
    print(result)


## 2. Add a resource and a prompt

Register a `text/plain` resource at `info://units` that explains the temperature units. Add a prompt named `explain_conversion` with a `celsius` argument. Retrieve both through the client.

Use the scaffold below. Replace its sample text and inspect the output. Explain why fetching the prompt does not produce an LLM answer.


In [ ]:
@practice_server.resource("info://units", mime_type="text/plain")
def units() -> str:
    """Describe the units used by this server."""
    return "Temperature input is in degrees Celsius."

@practice_server.prompt()
def explain_conversion(celsius: str) -> str:
    """Prepare a temperature conversion explanation request."""
    return f"Explain how to convert {celsius} degrees Celsius to Fahrenheit."

async with Client(practice_server) as client:
    print(await client.read_resource("info://units"))  # Read context, without calling an LLM.
    print(await client.get_prompt("explain_conversion", {"celsius": "20"}))


## 3. Inspect a server contract

Start `weather_mcp_server.py` in Terminal 1. Use the HTTP client below to list its tools. Inspect the `get_weather` schema: identify the required `city` argument, its type and description. Discovery requires no weather key.


In [ ]:
SERVER_URL = "http://127.0.0.1:8000/mcp"

async with Client(SERVER_URL) as client:
    for tool in await client.list_tools():  # Connect to the running weather server.
        print(tool)


## 4. Extend the chatbot

1. Run [mcp_chatbot.py](mcp_chatbot.py) in Terminal 2 after configuring the keys and starting the server in Terminal 1.
2. Ask about Perth, then Sydney. Identify the tool request and result messages.
3. Plan how to add a temperature conversion tool to a practice copy of the weather server. Identify the new schema, client call and host allowlist entry needed.
4. Explain which parts of the Function Calling loop remain unchanged.

## 5. Optional challenge: an arXiv tool

Reuse the arXiv request from Session 2. Plan a tool named `search_arxiv(query: str, max_results: int = 3)` that returns titles and links. Keep this exercise's server script and supporting files in this practice directory.

- Add a useful docstring and bounded result count.
- Register the function as a tool and inspect the generated schema.
- Test discovery and execution with a client before connecting it to a model.
- Add the tool name to the host allowlist, then ask the chatbot to search for papers.

## Reflection

Which changes belong in the server? Which belong in the host? How would you handle a tool that modifies business records rather than retrieving public data?
